# LangChain Use Cases - Groq Edition

**Stack:** Groq API + `llama-3.3-70b-versatile`  
**Original source:** [gkamradt/langchain-tutorials](https://github.com/gkamradt/langchain-tutorials)  
**Author:** Zariya Shahid  

This notebook covers the main real-world use cases of LangChain, rewritten from scratch using Groq instead of OpenAI.

**Use Cases covered:**
1. Summarization
2. Question and Answering Over Documents
3. Extraction
4. Chatbots with Memory
5. Agents with Tools
6. Interacting with APIs

## Setup

In [ ]:
# Install dependencies (run once)
!pip install langchain langchain-groq langchain-community langchain-text-splitters python-dotenv faiss-cpu sentence-transformers

In [ ]:
# Load API key from Colab Secrets
from google.colab import userdata
import os

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("Key loaded:", GROQ_API_KEY[:8], "...")

---
## 1. Summarization

Summarization lets you compress long text into key insights. Use cases include summarizing articles, research papers, meeting transcripts, and code documentation.

There are two approaches:
- **Short text** - simple prompt engineering
- **Long text** - manual map-reduce using LCEL

### 1.1 Short Text Summarization

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage

chat = ChatGroq(
    temperature=0,
    model_name="llama-3.3-70b-versatile",
    groq_api_key=GROQ_API_KEY
)

template = """
INSTRUCTIONS:
Please summarize the following text.
Respond in simple language that a beginner developer would understand.

TEXT:
{text}
"""

prompt = PromptTemplate(input_variables=["text"], template=template)

technical_text = """
Retrieval-Augmented Generation (RAG) is an AI framework that combines the strengths of
pre-trained large language models with external knowledge retrieval systems. Rather than
relying solely on information encoded in model weights during training, RAG dynamically
fetches relevant documents from an external corpus at inference time. The retrieved
documents are then prepended to the input prompt, providing the model with up-to-date
and domain-specific context to generate more accurate, grounded responses. This approach
reduces hallucinations, improves factual accuracy, and allows models to access information
beyond their training cutoff date.
"""

final_prompt = prompt.format(text=technical_text)
response = chat.invoke([HumanMessage(content=final_prompt)])
print(response.content)

### 1.2 Long Text Summarization with Map-Reduce

For large documents that exceed the context window, we split the text into chunks, summarize each chunk (map step), then summarize all the chunk summaries (reduce step).

In [ ]:
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

chat = ChatGroq(
    temperature=0,
    model_name="llama-3.3-70b-versatile",
    groq_api_key=GROQ_API_KEY
)

# Simulating a long document (in real projects, load from a file or URL)
long_text = """
Software engineering is the systematic application of engineering principles to the development,
operation, and maintenance of software. The field emerged in the late 1960s as a response to
the so-called software crisis, where projects frequently ran over budget, missed deadlines, or
produced unreliable results.

Modern software engineering encompasses a wide range of practices. Requirements engineering
involves understanding and documenting what stakeholders need from a system. System design
translates those requirements into an architecture. Implementation involves writing the actual
code using one or more programming languages. Testing ensures correctness, performance, and
security. Deployment makes the software available to users. Maintenance addresses bugs and
evolving requirements over time.

Agile methodologies have become dominant in the industry. Scrum organizes work into short
iterations called sprints. Kanban visualizes workflow on boards. Extreme Programming (XP)
emphasizes practices like test-driven development and pair programming. These methods prioritize
collaboration, customer feedback, and adaptability over rigid planning.

DevOps extends software engineering into operations, aiming to shorten the development lifecycle
and deliver features continuously. Continuous Integration (CI) automatically builds and tests
code on every commit. Continuous Deployment (CD) automatically releases tested code to production.
Infrastructure as Code (IaC) manages servers and environments through code rather than manual
configuration, enabling repeatability and version control.

Artificial intelligence is reshaping software engineering at every layer. AI-powered code
assistants like GitHub Copilot generate boilerplate and suggest completions. LLMs help with
documentation, bug detection, and code review. Increasingly, agentic systems can autonomously
plan, write, test, and deploy software given a high-level description of the desired outcome.
"""

# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n"],
    chunk_size=500,
    chunk_overlap=50
)
chunks = text_splitter.create_documents([long_text])
print(f"Split into {len(chunks)} chunks")

In [ ]:
# Map step: summarize each chunk individually
map_prompt = PromptTemplate.from_template(
    "Summarize this text in 1-2 sentences:\n\n{text}"
)
map_chain = map_prompt | chat | StrOutputParser()

chunk_summaries = [map_chain.invoke({"text": chunk.page_content}) for chunk in chunks]
print(f"Generated {len(chunk_summaries)} chunk summaries")

# Reduce step: combine all chunk summaries into one final summary
reduce_prompt = PromptTemplate.from_template(
    "The following are summaries of different sections of a document.\n"
    "Combine them into one coherent final summary in 3-4 sentences:\n\n{text}"
)
reduce_chain = reduce_prompt | chat | StrOutputParser()

combined = "\n\n".join(chunk_summaries)
final_summary = reduce_chain.invoke({"text": combined})

print("\nFinal Summary:\n", final_summary)

---
## 2. Question and Answering Over Documents (RAG)

The core pattern: embed your documents into a vectorstore, retrieve the most relevant chunks for a query, and pass them as context to the LLM.

**Pattern:** `query -> retrieve relevant chunks -> LLM(chunks + query) -> answer`

In [ ]:
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

chat = ChatGroq(
    temperature=0,
    model_name="llama-3.3-70b-versatile",
    groq_api_key=GROQ_API_KEY
)

# Knowledge base: LangChain components overview
knowledge_base = """
LangChain is a framework for building LLM-powered applications.
It provides components for prompts, models, memory, chains, and agents.

LCEL stands for LangChain Expression Language. It uses the pipe operator
to chain components together in a readable and composable way.
Example: chain = prompt | model | output_parser

Memory in LangChain allows conversation history to persist across turns.
ConversationBufferMemory stores all messages. ConversationSummaryMemory
stores a rolling summary to save tokens on long conversations.

Agents in LangChain use an LLM as a reasoning engine to decide which
tools to call. The agent receives the user input, reasons about which
tool to use, calls the tool, and uses the result to form a final response.

RAG stands for Retrieval Augmented Generation. It combines a retriever
that fetches relevant documents with a generator that produces the answer.
This allows LLMs to answer questions grounded in external knowledge.

ChromaDB and FAISS are popular vectorstores used with LangChain.
FAISS is best for local development. ChromaDB has more features for production.
"""

# Split and embed
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=40)
docs = text_splitter.create_documents([knowledge_base])

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)

print(f"Indexed {len(docs)} chunks into FAISS vectorstore.")

In [ ]:
# Build the RAG chain using LCEL
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

rag_prompt = ChatPromptTemplate.from_template(
    "Answer the question based only on the following context:\n\n"
    "{context}\n\n"
    "Question: {question}"
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | chat
    | StrOutputParser()
)

# Ask questions grounded in the knowledge base
questions = [
    "What is LCEL and how do you use it?",
    "What is the difference between FAISS and ChromaDB?",
    "How do agents work in LangChain?"
]

for q in questions:
    answer = rag_chain.invoke(q)
    print(f"Q: {q}")
    print(f"A: {answer}\n")

---
## 3. Extraction

Extraction means pulling structured data out of unstructured text. Instead of getting a paragraph back, you get a clean Python dictionary you can actually use in your code.

**Pattern:** `raw text -> LLM with format instructions -> structured dict`

### 3.1 Simple Extraction

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

chat = ChatGroq(
    temperature=0,
    model_name="llama-3.3-70b-versatile",
    groq_api_key=GROQ_API_KEY
)

# Extract tech stack from a job description
instructions = """
You will be given a job description. Extract the required technologies and return them
as a Python dictionary with keys: 'languages', 'frameworks', 'databases', 'tools'.
Return only the dictionary, no other text.
"""

job_description = """
We are looking for a Full Stack Engineer to join our team.
You should be proficient in React and Next.js for the frontend.
On the backend, we use Node.js with Express. Our primary database is PostgreSQL,
and we use Redis for caching. The team uses Docker for containerization and
GitHub Actions for CI/CD. Familiarity with TypeScript is a strong plus.
"""

response = chat.invoke([HumanMessage(content=instructions + job_description)])
print("Raw output:")
print(response.content)

In [ ]:
# Convert to a real Python dict
import ast

tech_stack = ast.literal_eval(response.content)
print("Parsed tech stack:", tech_stack)
print("\nFrameworks required:", tech_stack.get("frameworks"))

### 3.2 Structured Extraction with Pydantic and JsonOutputParser

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field

# Define what we want to extract
class PersonInfo(BaseModel):
    name: str = Field(description="Full name of the person")
    role: str = Field(description="Their job title or role")
    skills: str = Field(description="List of technical skills mentioned, as a comma-separated string")
    years_experience: str = Field(description="Years of experience if mentioned, otherwise 'Not specified'")

output_parser = JsonOutputParser(pydantic_object=PersonInfo)
format_instructions = output_parser.get_format_instructions()

prompt = PromptTemplate(
    template="Extract information from this text.\n{format_instructions}\n\nText: {text}",
    input_variables=["text"],
    partial_variables={"format_instructions": format_instructions}
)

bio = """
Ali Hassan is a backend engineer with 4 years of experience.
He specializes in Python, FastAPI, PostgreSQL, and Docker.
He recently started learning Kubernetes and LangChain.
"""

chain = prompt | chat | output_parser
parsed = chain.invoke({"text": bio})

print("Extracted data:")
for key, value in parsed.items():
    print(f"  {key}: {value}")

---
## 4. Chatbot with Memory

A chatbot needs to remember what was said earlier in the conversation. Without memory, every message is treated as a fresh start.

We manually maintain a message history list and pass it to the model on every turn.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

chat = ChatGroq(
    temperature=0.7,
    model_name="llama-3.3-70b-versatile",
    groq_api_key=GROQ_API_KEY
)

# Start with a system message that sets the persona
chat_history = [
    SystemMessage(content="You are a helpful technical mentor for a software engineering student. Answer concisely and encourage learning.")
]

def chat_with_memory(user_input: str) -> str:
    """Send a message and keep the full history for context."""
    chat_history.append(HumanMessage(content=user_input))
    response = chat.invoke(chat_history)
    chat_history.append(AIMessage(content=response.content))
    return response.content

In [ ]:
# Turn 1
response1 = chat_with_memory("What is LangChain and why should I learn it?")
print("Turn 1:", response1)

In [ ]:
# Turn 2 - references the previous answer
response2 = chat_with_memory("What did you just say were the main benefits?")
print("Turn 2:", response2)

In [ ]:
# Turn 3
response3 = chat_with_memory("Can you give me a one-week learning plan for it?")
print("Turn 3:", response3)

In [ ]:
# Inspect the full conversation history
print("Full conversation history:")
for msg in chat_history:
    print(f"  [{msg.type}]: {msg.content[:80]}...")

---
## 5. Agents with Tools

Agents let the LLM decide which tool to call based on the user's input. The LLM reasons step-by-step (ReAct pattern) and stops when it has a final answer.

**Pattern:** `user input -> LLM reasons -> picks tool -> gets result -> reasons again -> final answer`

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
import math

chat = ChatGroq(
    temperature=0,
    model_name="llama-3.3-70b-versatile",
    groq_api_key=GROQ_API_KEY
)

# Define tools the agent can use
@tool
def calculate_bmi(weight_kg: float, height_m: float) -> str:
    """Calculates BMI given weight in kg and height in meters. Returns BMI value and category."""
    bmi = weight_kg / (height_m ** 2)
    if bmi < 18.5:
        category = "Underweight"
    elif bmi < 25:
        category = "Normal weight"
    elif bmi < 30:
        category = "Overweight"
    else:
        category = "Obese"
    return f"BMI: {bmi:.2f}, Category: {category}"

@tool
def celsius_to_fahrenheit(celsius: float) -> str:
    """Converts a temperature from Celsius to Fahrenheit."""
    fahrenheit = (celsius * 9/5) + 32
    return f"{celsius}C = {fahrenheit:.1f}F"

@tool
def count_words(text: str) -> str:
    """Counts the number of words in a given text string."""
    count = len(text.split())
    return f"The text has {count} words."

tools = [calculate_bmi, celsius_to_fahrenheit, count_words]

# Create the agent using LangGraph (recommended in LangChain 1.x)
agent = create_react_agent(chat, tools)
print("Agent created successfully.")

In [ ]:
# The agent decides which tool to call
result = agent.invoke({"messages": [{"role": "user", "content": "My weight is 70 kg and my height is 1.75 m. What is my BMI?"}]})
print("\nFinal Answer:", result["messages"][-1].content)

In [ ]:
# Agent handles a different tool
result = agent.invoke({"messages": [{"role": "user", "content": "Today in Faisalabad it is 42 degrees Celsius. What is that in Fahrenheit?"}]})
print("\nFinal Answer:", result["messages"][-1].content)

In [ ]:
# Agent decides without being told which tool to use
result = agent.invoke({"messages": [{"role": "user", "content": "How many words are in this sentence: 'LangChain makes building AI applications easier and faster'"}]})
print("\nFinal Answer:", result["messages"][-1].content)

---
## 6. Interacting with APIs

LLMs can read API documentation and figure out which endpoint to call. Here we demonstrate parsing a user query and constructing an API call manually, a clean and lightweight pattern without extra dependencies.

In [ ]:
import requests
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

chat = ChatGroq(
    temperature=0,
    model_name="llama-3.3-70b-versatile",
    groq_api_key=GROQ_API_KEY
)

# Step 1: Use LLM to extract the country name from a natural language query
class CountryExtraction(BaseModel):
    country: str = Field(description="The country name mentioned in lowercase, e.g. 'pakistan', 'france'")

output_parser = JsonOutputParser(pydantic_object=CountryExtraction)
format_instructions = output_parser.get_format_instructions()

template = """
Extract the country name from this user query.
{format_instructions}

User query: {query}
"""

prompt = PromptTemplate(
    input_variables=["query"],
    partial_variables={"format_instructions": format_instructions},
    template=template
)

user_query = "Can you tell me about the currency and capital of Pakistan?"

chain = prompt | chat | output_parser
parsed = chain.invoke({"query": user_query})
country = parsed["country"]
print(f"Extracted country: {country}")

In [ ]:
# Step 2: Make the actual API call
api_url = f"https://restcountries.com/v3.1/name/{country}"
response = requests.get(api_url)
data = response.json()[0]

# Extract relevant fields
capital = data.get("capital", ["Unknown"])[0]
currencies = data.get("currencies", {})
currency_info = ", ".join([f"{v['name']} ({k})" for k, v in currencies.items()])
population = data.get("population", "Unknown")
region = data.get("region", "Unknown")

print(f"Country: {data['name']['common']}")
print(f"Capital: {capital}")
print(f"Currency: {currency_info}")
print(f"Population: {population:,}")
print(f"Region: {region}")

In [ ]:
# Step 3: Use the LLM to generate a natural language response from the API data
summary_prompt = f"""
Based on this data, answer the user's question in 2-3 sentences.

User question: {user_query}

Data:
- Capital: {capital}
- Currency: {currency_info}
- Population: {population:,}
- Region: {region}
"""

final_response = chat.invoke([HumanMessage(content=summary_prompt)])
print("\nFinal Response:")
print(final_response.content)

---
## Summary

| Use Case | Key Components Used |
|---|---|
| Summarization (short) | PromptTemplate + ChatGroq |
| Summarization (long) | RecursiveCharacterTextSplitter + LCEL map-reduce |
| Q&A over Documents | FAISS + HuggingFaceEmbeddings + LCEL RAG chain |
| Extraction | JsonOutputParser + Pydantic + LCEL |
| Chatbot with Memory | Manual message history list + ChatGroq |
| Agents | create_react_agent (LangGraph) + @tool |
| API Interaction | LLM for intent parsing + requests for API call + LLM for response formatting |

**Next up:** Step 2 - Memory and Conversation Chains in depth.